## Data processing
In questo notebook andrò a processare, per ogni corso e per il dataset già concatenato,
la colonna timeseries "esplodendo" la serie temporale attualmente densa in un array 2D,
nominando anche le singole azioni.

Gli output verranno salvati in /data/processed/ per le successive aggregazioni con variabili
LAG

In [1]:
import os
import numpy as np
import pandas as pd
import ast

In [2]:
# Data and output paths

INPUT_DIR = "/datasets/unitelma"
OUTPUT_DIR = "/notebooks/data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# List of ACTIONS

ACTIONS = np.array(['view','write','user report','update','view forum',
    'subscribe','view forums','view discussion','add discussion','search',
    'view all','unsubscribe','subscribeall','add','add contact',
    'remove contact','history','block contact','update post','pre-view',
    'launch recent','add post','error','unblock contact',
    'automatically create user token','sending requested user token',
    'core_webservice_get_site_info','ltol_get_categories',
    'core_enrol_get_users_courses','launch','ltol_ws_ltol_get_courses',
    'core_course_get_contents', 'ltol_ws_get_asset','ltol_ws_put_track','talk',
    'report', 'attempt', 'continue attempt','view summary','close attempt',
    'review', 'edit', 'mark read','ltol_get_users_courses','stop tracking',
    'start tracking','view subscriber','upload','unsubscribeall','assign',
    'enrol','delete discussion','delete post','editvideos','mail blocked',
    'delete','trk: l2lscorm at: 9','comment','map','open','add entry',
    'trk: l2lscorm at: 1','diff','report outline','report log','edit entry',
    'trk: l2lscorm at: 3','trk: l2lscorm at: 2','add page','flag',
    'trk: l2lscorm at: 4','created','sent','called','searched','uploaded',
    'updated','downloaded','launched','graded','submitted','reset','started',
    'reviewed','failed','added','deleted','blocked','unblocked','disabled',
    'removed','accepted','assigned','restored','unassigned','abandoned',
    'recent'])

In [16]:
def process_file(file_path, course_id):
    """
    Legge un csv, esegue il parsing della colonna 'timeseries' e esplode i dati 
    creando un record per ogni studente e giorno.
    """
    print(f"[LOG] Processing: {os.path.basename(file_path)}...")
    df = pd.read_csv(file_path)
    
    expanded_rows = []
    
    for idx, row in df.iterrows():
        student_id = row['student_id']
        dropout = row['dropout']
        
        # Converte la stringa della lista in una lista di liste di interi
        ts_data = ast.literal_eval(row['timeseries'])
        
        # Ogni iterazione rappresenta un "giorno" (timestep)
        for day_index, daily_actions in enumerate(ts_data):
            # Assicuro che daily_actions abbia dimensione 97
            # Costruiamo il record
            record = {
                'student_id': student_id,
                'course_id': course_id,
                'day': day_index + 1,
                'dropout': dropout
            }
            # Popola le azioni per quel giorno
            for action_name, action_count in zip(ACTIONS, daily_actions):
                record[action_name] = action_count
                
            expanded_rows.append(record)
            
    df_expanded = pd.DataFrame(expanded_rows)
    return df_expanded


In [17]:
for i in range(1, 14):
    file_name = f"timeseries_{i}.csv"
    input_path = os.path.join(INPUT_DIR, file_name)
    output_path = os.path.join(OUTPUT_DIR, f"processed_timeseries_{i}.csv")
    
    if os.path.exists(input_path):
        df_proc = process_file(input_path, course_id=i)
        
        
        df_proc.to_csv(output_path, index=False)
        print(f"[LOG] Saved to {output_path} (Shape: {df_proc.shape})")
    else:
        print(f"[ERROR] File not found: {input_path}")

print("[LOG] Preprocessing compleTE")

[LOG] Processing: timeseries_1.csv...
[LOG] Saved to /notebooks/data/processed/processed_timeseries_1.csv (Shape: (40680, 101))
[LOG] Processing: timeseries_2.csv...
[LOG] Saved to /notebooks/data/processed/processed_timeseries_2.csv (Shape: (90000, 101))
[LOG] Processing: timeseries_3.csv...
[LOG] Saved to /notebooks/data/processed/processed_timeseries_3.csv (Shape: (27000, 101))
[LOG] Processing: timeseries_4.csv...
[LOG] Saved to /notebooks/data/processed/processed_timeseries_4.csv (Shape: (243360, 101))
[LOG] Processing: timeseries_5.csv...
[LOG] Saved to /notebooks/data/processed/processed_timeseries_5.csv (Shape: (96840, 101))
[LOG] Processing: timeseries_6.csv...
[LOG] Saved to /notebooks/data/processed/processed_timeseries_6.csv (Shape: (168840, 101))
[LOG] Processing: timeseries_7.csv...
[LOG] Saved to /notebooks/data/processed/processed_timeseries_7.csv (Shape: (73440, 101))
[LOG] Processing: timeseries_8.csv...
[LOG] Saved to /notebooks/data/processed/processed_timeseries_8.

## Caso all_timeseries.csv
Concatenazione di tutti i corsi in un unico file

In [5]:
all_dfs = []

print("[LOG] Creazione del dataset aggregato dai file dei singoli corsi...")

# Iteriamo sui 13 file appena creati
for i in range(1, 14):
    file_path = os.path.join(OUTPUT_DIR, f"processed_timeseries_{i}.csv")
    
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        all_dfs.append(df)
    else:
        print(f"[WARNING] File non trovato, impossibile concatenare: {file_path}")

# Concatenazione e salvataggio
if all_dfs:
    # Unisce tutti i dataframe in uno solo
    df_all = pd.concat(all_dfs, ignore_index=True)
    
    output_path_all = os.path.join(OUTPUT_DIR, "processed_all_timeseries.csv")
    df_all.to_csv(output_path_all, index=False)
    
    print(f"[OK] Salvato dataset concatenato in {output_path_all} - Shape: {df_all.shape}")
else:
    print("[ERROR] Nessun file trovato per la concatenazione.")

[LOG] Creazione del dataset aggregato dai file dei singoli corsi...
[OK] Salvato dataset concatenato in /notebooks/data/processed/processed_all_timeseries.csv - Shape: (1856160, 101)


In [ ]:
# TODO change paths